In [ ]:
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
import re

In [ ]:
import sys, os

# AutomationModule lives in the root of the tidy3d project
sys.path.append(os.path.abspath(r'H:\codes\tidy3d'))
import AutomationModule as AM


In [ ]:
tidy3dAPI = os.environ["API_TIDY3D_KEY"]


In [ ]:
lambdas = np.array([4,2.2])
run = True


params example:{'aspect_ratio': 1,
 'background_permittivity': 1.0,
 'box_size': array([12.        , 12.        ,  8.48528137]),
 'd': 1.2,
 'defect_density': 0.09984609641754491,
 'dz': 1.697056274847714,
 'ff': 0.450616519434629,
 'ff_analytic': 0.4101477321755692,
 'grid_size': array([400, 400, 283]),
 'kappa': 2.8,
 'major_radius': 0.25781249999999994,
 'minor_radius': 0.25781249999999994,
 'permittivity': 6.25,
 'seed': 12345}

In [ ]:
project_name = "20260918_Transmission_Experiment_woodpiles"
folder_path = rf"../Structures"
direction = "z" 
runtime_ps = 22e-12
min_steps_per_lambda = 18
for dirpath, dirnames, filenames in os.walk(folder_path):
    if "Large_Structures" in dirpath:
        continue
    for filename in filenames:
        ref = True
        if filename.endswith(".h5"):
            # print(f"Processing file: {filename}")
            if os.path.isfile(os.path.join(dirpath, filename)):
                file=os.path.join(dirpath, filename)
                params = AM.read_hdf5_as_dict(file)["params"]
                structure_1 = AM.loadAndRunStructure(key = tidy3dAPI, file_path=file
                                                     ,direction=direction, lambda_range=lambdas,
                                                     box_size=list(params["box_size"]),runtime_ps=runtime_ps,min_steps_per_lambda=min_steps_per_lambda,
                                                    scaling=1,shuoff_condtion=1e-20, verbose=True, 
                                                    monitors=["flux"],
                                                    freqs=300, 
                                                     source="planewave", absorbers=130,sim_name=rf"{Path(filename).stem}"
                                                    )
                if run:
                    id0=""
                    sim=structure_1.sim
                    folder_desc = rf"{Path.cwd().parents[2]}/data/{project_name}/n_{np.sqrt(params['permittivity']):.2f}"
                    os.makedirs(folder_desc, exist_ok=True)
                    sim_name=rf"{Path(filename).stem}"
                    if os.path.exists(os.path.join(folder_desc, sim_name+".txt")):
                        print("Exist!",os.path.join(folder_desc, sim_name+".txt"))
                        continue
                    else:
                        print("RUN ",os.path.join(folder_desc, sim_name+".txt"))
                        if ref:
                            sim0=sim.copy(update={"structures":[]})
                            id0 =web.upload(sim0, folder_name=project_name,task_name=sim_name+"_0", verbose=True)
                            web.start(task_id = id0)
                            web.monitor(id0)
                            ref=False

                        id =web.upload(sim, folder_name=project_name,task_name=sim_name, verbose=True)
                        ids = id0+'\n' + id
                        with open(os.path.join(folder_desc, sim_name+".txt"), "w") as file:
                            # Write the string to the file
                            file.write(ids)
                        web.start(task_id = id)
                        web.monitor(id)
                else: 
                    structure_1.plot_sim_layout()

       